# 07 – Modeling IS_WIN avec MLflow

Ce notebook charge le dernier dataset gold, entraîne plusieurs modèles (LightGBM, XGBoost, stacking) avec logging complet dans MLflow, et affiche un résumé des métriques. Lance `mlflow ui` dans un autre terminal pour explorer les runs.

In [1]:
import pandas as pd

from src.modeling import DatasetConfig, TrainingConfig, ModelRunConfig, ModelTrainer

dataset_cfg = DatasetConfig(target="IS_WIN")
training_cfg = TrainingConfig(
    experiment_name="is_win_baseline",
    tracking_uri="file:./mlruns",
    test_size=0.2,
    random_state=42,
    enable_learning_curve=False,
)

trainer = ModelTrainer(dataset_cfg, training_cfg)
print("Dataset loaded from:", dataset_cfg.resolve_path())
print("Train shape:", trainer.X_train.shape, "- Test shape:", trainer.X_test.shape)

Dataset loaded from: data/03_gold/gold_dataset_is_win_20251110T221340Z.parquet
Train shape: (25792, 280) - Test shape: (6449, 280)


## Distribution de la cible

In [2]:
target_distribution = pd.Series(trainer.y_train, name="IS_WIN").value_counts(normalize=True)
display(target_distribution)

IS_WIN
1    0.5884
0    0.4116
Name: proportion, dtype: float64

## Entraînement & logging MLflow

In [3]:
results_df = trainer.train_models(["lgbm", "xgb", "stacking"])
display(results_df)

[LightGBM] [Info] Number of positive: 15176, number of negative: 10616
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002959 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12914
[LightGBM] [Info] Number of data points in the train set: 25792, number of used features: 103
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.588400 -> initscore=0.357353
[LightGBM] [Info] Start training from score 0.357353


/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2025/11/10 23:20:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/10 23:20:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/10 23:20:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/10 23:20:37 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when log

[LightGBM] [Info] Number of positive: 15176, number of negative: 10616
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.066845 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12914
[LightGBM] [Info] Number of data points in the train set: 25792, number of used features: 103
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.588400 -> initscore=0.357353
[LightGBM] [Info] Start training from score 0.357353
[LightGBM] [Info] Number of positive: 12141, number of negative: 8493
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007278 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12804
[LightGBM] [Info] Number of data points in the train set: 20634, number of used features: 103
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.588398 -> initscore=0.357346
[L

/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWa

,model,accuracy,precision,recall,f1,roc_auc,log_loss,brier
0,lgbm,0.656226,0.680311,0.784133,0.728542,0.694153,0.620086,0.215479
1,xgb,0.657621,0.679737,0.790459,0.730929,0.691419,0.624021,0.216855
2,stacking,0.662428,0.679627,0.806273,0.737553,0.694781,0.619394,0.215169


Chaque run enregistre les métriques, courbes ROC/calibration, matrices de confusion et importances de features dans MLflow (`mlruns`).